# DuckDB and pandas Working Together

DuckDB and pandas complement each other:

- **pandas** is convenient for Python-oriented data preparation, exploration, and integration with the Python ecosystem.
- **DuckDB** provides analytical SQL, joins, aggregations, and direct Parquet access.
- DuckDB can query a pandas DataFrame without first copying it into a permanent database table.
- DuckDB query results can be returned as pandas DataFrames.

This notebook uses the MovieLens data from the previous DuckDB example and writes the popular-movies result in two ways:

1. With `DataFrame.to_parquet()`
2. With DuckDB's `COPY ... TO` statement

## Installation

```powershell
python -m pip install pandas pyarrow duckdb
```


## 1. Imports and paths

The source MovieLens files are Parquet. Both generated result files are written to `C:\data\output`.


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

MOVIELENS_PARQUET_DIR = Path(r"C:\data\movielens\parquet")
MOVIES_PATH = MOVIELENS_PARQUET_DIR / "movies.parquet"
RATINGS_PATH = MOVIELENS_PARQUET_DIR / "ratings.parquet"

OUTPUT_DIR = Path(r"C:\data\output")
PANDAS_OUTPUT_PATH = OUTPUT_DIR / "popular-movies-using-pandas.parquet"
DUCKDB_OUTPUT_PATH = OUTPUT_DIR / "popular-movies-using-duckdb.parquet"

for source_path in (MOVIES_PATH, RATINGS_PATH):
    if not source_path.is_file():
        raise FileNotFoundError(f"Required source file not found: {source_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"pandas version: {pd.__version__}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Parquet → pandas

Pandas reads the two MovieLens Parquet files into DataFrames. This is useful when subsequent work needs pandas-specific operations or another Python library that accepts DataFrames.


In [ ]:
movies_df = pd.read_parquet(MOVIES_PATH, engine="pyarrow")
ratings_df = pd.read_parquet(RATINGS_PATH, engine="pyarrow")

print(f"movies_df: {movies_df.shape[0]:,} rows × {movies_df.shape[1]} columns")
print(movies_df.head())

print(f"\nratings_df: {ratings_df.shape[0]:,} rows × {ratings_df.shape[1]} columns")
print(ratings_df.head())


## 3. Query a pandas DataFrame with DuckDB

DuckDB supports **replacement scans**. When a SQL table name matches a DataFrame variable in the Python scope, DuckDB can scan that DataFrame directly.

This is convenient for short interactive work. For longer workflows, explicit registration makes the SQL-to-DataFrame mapping clearer.


In [ ]:
replacement_scan_preview = duckdb.sql("""
    SELECT
        movieId,
        count(*) AS rating_count,
        round(avg(rating), 3) AS average_rating
    FROM ratings_df
    GROUP BY movieId
    ORDER BY rating_count DESC, movieId
    LIMIT 5
""").df()

print(replacement_scan_preview)


## 4. Explicitly register DataFrames

`connection.register(name, dataframe)` exposes an existing DataFrame as a temporary DuckDB view. It does not create a permanent database table.

Explicit registration is recommended when stable names and connection-local behavior matter.


In [ ]:
connection = duckdb.connect(database=":memory:")
connection.register("movies_from_pandas", movies_df)
connection.register("ratings_from_pandas", ratings_df)

registered_objects = connection.execute("SHOW TABLES").fetchall()
print("Objects visible to DuckDB:")
for item in registered_objects:
    print(item[0])


## 5. pandas → DuckDB → pandas

The SQL query reads the registered pandas DataFrames. Calling `.df()` converts the DuckDB result into a new pandas DataFrame.

The popular-movies rule matches the previous notebook:

- at least 100 distinct users;
- average rating of 4.0 or higher;
- join to movie metadata for title and genres.


In [ ]:
popular_movies_sql = """
WITH rating_summary AS (
    SELECT
        movieId,
        count(*) AS rating_count,
        count(DISTINCT userId) AS distinct_user_count,
        avg(rating) AS average_rating
    FROM ratings_from_pandas
    GROUP BY movieId
    HAVING count(DISTINCT userId) >= 100
       AND avg(rating) >= 4.0
)
SELECT
    summary.movieId,
    movies.title,
    movies.genres,
    summary.rating_count,
    summary.distinct_user_count,
    round(summary.average_rating, 3) AS average_rating
FROM rating_summary AS summary
INNER JOIN movies_from_pandas AS movies
    ON summary.movieId = movies.movieId
ORDER BY
    average_rating DESC,
    summary.distinct_user_count DESC,
    summary.movieId
"""

popular_movies_df = connection.execute(popular_movies_sql).df()

print(popular_movies_df.head(10))
print(f"\nResult type: {type(popular_movies_df).__name__}")
print(f"Popular movies: {len(popular_movies_df)}")

assert isinstance(popular_movies_df, pd.DataFrame)
assert not popular_movies_df.empty
assert popular_movies_df["distinct_user_count"].min() >= 100
assert popular_movies_df["average_rating"].min() >= 4.0


## 6. Write the result with pandas

The first output uses pandas and PyArrow. `index=False` prevents the default DataFrame index from becoming a stored data column.


In [ ]:
popular_movies_df.to_parquet(
    PANDAS_OUTPUT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

print(f"Created with pandas: {PANDAS_OUTPUT_PATH}")
print(f"File size: {PANDAS_OUTPUT_PATH.stat().st_size:,} bytes")


## 7. Write the same DataFrame with DuckDB

The result DataFrame is registered back into DuckDB. DuckDB then writes it with `COPY`.

This completes both interoperability directions:

```text
pandas DataFrame → DuckDB SQL → pandas DataFrame → DuckDB
```


In [ ]:
connection.register("popular_movies_from_pandas", popular_movies_df)

duckdb_output_sql = "'" + str(DUCKDB_OUTPUT_PATH).replace("'", "''") + "'"
connection.execute(f"""
    COPY (
        SELECT *
        FROM popular_movies_from_pandas
        ORDER BY average_rating DESC, distinct_user_count DESC, movieId
    )
    TO {duckdb_output_sql}
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Created with DuckDB: {DUCKDB_OUTPUT_PATH}")
print(f"File size: {DUCKDB_OUTPUT_PATH.stat().st_size:,} bytes")


## 8. Read and compare both outputs

Pandas reads the pandas-written file. DuckDB reads the DuckDB-written file and returns it as a DataFrame. The two results should contain the same values in the same order.


In [ ]:
pandas_written_df = pd.read_parquet(PANDAS_OUTPUT_PATH, engine="pyarrow")

duckdb_output_sql = "'" + str(DUCKDB_OUTPUT_PATH).replace("'", "''") + "'"
duckdb_written_df = connection.execute(f"""
    SELECT *
    FROM read_parquet({duckdb_output_sql})
    ORDER BY average_rating DESC, distinct_user_count DESC, movieId
""").df()

pd.testing.assert_frame_equal(
    pandas_written_df.reset_index(drop=True),
    duckdb_written_df.reset_index(drop=True),
    check_dtype=False,
)

print("Both Parquet outputs contain the same data.")
print(f"Rows in each output: {len(pandas_written_df)}")
print(f"pandas output size: {PANDAS_OUTPUT_PATH.stat().st_size:,} bytes")
print(f"DuckDB output size: {DUCKDB_OUTPUT_PATH.stat().st_size:,} bytes")


## 9. Clean up the in-memory connection


In [ ]:
connection.close()
print("DuckDB connection closed.")


## Summary

- `pd.read_parquet()` loads Parquet into pandas.
- DuckDB replacement scans can query a DataFrame variable directly.
- `connection.register()` explicitly exposes a DataFrame to DuckDB SQL.
- DuckDB's `.df()` returns a query result as a pandas DataFrame.
- A pandas result can be registered back into DuckDB.
- Both pandas and DuckDB can write Parquet files.

Generated files:

- `C:\data\output\popular-movies-using-pandas.parquet`
- `C:\data\output\popular-movies-using-duckdb.parquet`
